In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio httpx kailash-ml kailash-align kailash-kaizen

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# M6 Ollama bootstrap — local LLMs, no API keys
# On Colab: installs + starts Ollama and pulls this lesson's models.
# Locally: verifies the daemon is running and models are pulled.
# ══════════════════════════════════════════════════════════════════
_LESSON_MODELS = ['qwen2.5:0.5b', 'llama3.2:3b']
import sys, os, time, shutil, subprocess
import httpx

_OLLAMA_BASE_URL = os.environ.get('OLLAMA_BASE_URL', 'http://localhost:11434')
_IN_COLAB = 'google.colab' in sys.modules

def _daemon_up(timeout_s=1.0):
    try:
        r = httpx.get(f'{_OLLAMA_BASE_URL}/api/tags', timeout=timeout_s)
        r.raise_for_status()
        return r.json()
    except Exception:
        return None

if _IN_COLAB and shutil.which('ollama') is None:
    print('[ollama] installing binary …')
    subprocess.run(
        'curl -fsSL https://ollama.com/install.sh | sh',
        shell=True, check=True,
    )

if _IN_COLAB and _daemon_up() is None:
    print('[ollama] starting daemon …')
    subprocess.Popen(
        'nohup ollama serve > /tmp/ollama.log 2>&1 &',
        shell=True,
    )
    deadline = time.monotonic() + 30
    while time.monotonic() < deadline and _daemon_up() is None:
        time.sleep(0.5)

_tags = _daemon_up(timeout_s=2.0)
if _tags is None:
    raise RuntimeError(
        f'Ollama daemon not reachable at {_OLLAMA_BASE_URL}. '
        'On Colab: re-run this cell. Locally: run `ollama serve`.'
    )

_have = {m.get('name','').split(':',1)[0] for m in _tags.get('models', [])}
for _model in _LESSON_MODELS:
    _fam = _model.split(':', 1)[0]
    if _fam in _have:
        continue
    if _IN_COLAB:
        print(f'[ollama] pulling {_model} (one-time) …')
        subprocess.run(['ollama', 'pull', _model], check=True)
    else:
        raise RuntimeError(
            f'Model {_model!r} not pulled. Run: ollama pull {_model}'
        )

print(
    '✓ Ollama ready at', _OLLAMA_BASE_URL,
    '— models:', _LESSON_MODELS,
)



In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp06
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import logging
import os
from dataclasses import dataclass, field, replace
from pathlib import Path
from typing import Any, Literal

import numpy as np
import polars as pl
from dotenv import load_dotenv

logger = logging.getLogger(__name__)


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Return the best available compute device as a ``torch.device``.

    Routes through ``kailash_ml.device()`` (the canonical kailash-ml 0.12+
    accelerator detector) so MPS / CUDA / ROCm / Intel XPU / CPU are all
    selected by the same logic the rest of the platform uses. The previous
    hand-rolled ``mps→cuda→cpu`` cascade is replaced because:

      * kailash-ml's detector knows about ROCm, Intel XPU, and fp16/bf16
        capability flags — the cascade in this helper did not.
      * Apple-Silicon students get the Metal Performance Shaders backend
        with mixed-precision (fp16) without any opt-in.
      * One detection point means lessons that print "Using device: …"
        agree with what kailash-ml's MLEngine() actually picks.
    """
    import kailash_ml as km
    import torch

    backend = km.device()  # BackendInfo (auto MPS on Mac, CUDA on Linux+NVIDIA, …)
    return torch.device(backend.device_string)


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ════════════════════════════════════════════════════════════════════════
# SPLIT FIRST, THEN PREPROCESS
# ════════════════════════════════════════════════════════════════════════
# kailash-ml's ``PreprocessingPipeline.setup()`` fits its imputer, its
# categorical encoder (including TARGET encoding) and its scaler on EVERY
# row it is given, and only afterwards splits train/test. Handing it the
# whole dataset therefore lets test-set statistics leak into training.
#
# The helpers below hold the test rows out FIRST, fit the pipeline on the
# training rows only, and push the held-out rows through the fitted
# ``transform()``. ``setup()`` still splits internally, but that split now
# happens inside the training rows, so it cannot leak anything.

_SETUP_OPTIONS: frozenset[str] = frozenset(
    {
        "normalize",
        "normalize_method",
        "categorical_encoding",
        "imputation_strategy",
        "impute_n_neighbors",
        "remove_outliers",
        "outlier_threshold",
        "max_cardinality",
        "exclude_columns",
        "remove_multicollinearity",
        "multicollinearity_threshold",
        "pca",
        "pca_components",
        "transform_target",
        "fix_imbalance",
        "imbalance_method",
    }
)

# Arguments the helpers own themselves; passing them through would
# re-introduce the leak (``data`` = the full frame) or fight the split.
_OWNED_BY_HELPER: frozenset[str] = frozenset({"data", "target", "train_size", "seed"})


@dataclass(frozen=True)
class PreprocessResult:
    """Outcome of :func:`split_then_preprocess` / :func:`preprocess_train_test`.

    Mirrors the fields of kailash-ml's ``SetupResult`` that exercises use,
    plus the fitted pipeline and a record of what was done to which split.

    ``train_data`` / ``test_data``
        Both produced by the SAME pipeline, fitted on the training rows only.
    ``transformers``
        The fitted imputer / encoder / scaler state (e.g.
        ``transformers["ordinal_mappings"]``), learned from training rows.
    ``train_only_steps``
        Steps applied to the training split only (outlier removal,
        class-imbalance resampling). The test split is never filtered or
        resampled — it stays a faithful sample of the data.
    ``test_null_fills``
        Columns that were complete in the training rows but have gaps in
        the test rows (the fitted pipeline has no learned fill for them),
        mapped to the TRAINING statistic used to fill them.
    """

    train_data: pl.DataFrame
    test_data: pl.DataFrame
    target_column: str
    task_type: str
    numeric_columns: list[str]
    categorical_columns: list[str]
    transformers: dict[str, Any]
    pipeline: Any
    original_shape: tuple[int, int]
    train_raw_rows: int
    test_raw_rows: int
    stratified: bool
    train_only_steps: tuple[str, ...] = ()
    test_null_fills: dict[str, Any] = field(default_factory=dict)
    summary: str = ""


def _looks_like_classification(series: pl.Series) -> bool:
    """Same rule kailash-ml uses to call a target "classification"."""
    if series.dtype in (pl.Boolean, pl.Categorical, pl.Utf8, pl.String):
        return True
    return series.drop_nulls().n_unique() <= 20


def split_raw_train_test(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
) -> tuple[pl.DataFrame, pl.DataFrame, bool]:
    """Hold out test rows from the RAW frame before anything is fitted.

    Rows are shuffled with ``seed``. ``test_size`` is a fraction in (0, 1)
    or an exact row count. With ``stratify="auto"`` a classification target
    (same rule as kailash-ml) is stratified so every class keeps its share
    in both splits; ``True`` / ``False`` force the choice.

    Returns ``(train_raw, test_raw, stratified)``; both splits keep the
    shuffled order.
    """
    if target not in df.columns:
        raise ValueError(f"Target column '{target}' not found in data.")
    n = df.height
    if n < 2:
        raise ValueError(f"Need at least 2 rows to split, got {n}.")
    if isinstance(test_size, np.integer):
        test_size = int(test_size)
    if isinstance(test_size, bool) or not isinstance(test_size, (int, float)):
        raise TypeError(
            f"test_size must be a fraction in (0, 1) or a row count, got {test_size!r}."
        )
    if isinstance(test_size, int):
        if not 1 <= test_size < n:
            raise ValueError(f"test_size as a row count must be in [1, {n - 1}], got {test_size}.")
        test_fraction = test_size / n
    else:
        if not 0.0 < test_size < 1.0:
            raise ValueError(f"test_size as a fraction must be in (0, 1), got {test_size}.")
        test_fraction = float(test_size)

    is_classification = _looks_like_classification(df[target])
    if stratify == "auto":
        stratified = is_classification
    elif stratify is True:
        if not is_classification:
            raise ValueError(
                f"stratify=True needs a classification target; '{target}' has "
                f"{df[target].drop_nulls().n_unique()} distinct values."
            )
        stratified = True
    elif stratify is False:
        stratified = False
    else:
        raise TypeError(f"stratify must be True, False or 'auto', got {stratify!r}.")

    perm = np.random.RandomState(seed).permutation(n)
    shuffled = df[perm.tolist()]

    if stratified:
        # Within each class (in shuffled order), the first round(n_c * f)
        # rows go to test — the class share is the same in both splits.
        is_test = pl.int_range(pl.len()).over(target) < (
            pl.len().over(target) * test_fraction
        ).round()
        flagged = shuffled.with_columns(is_test.alias("__split_is_test"))
        test_raw = flagged.filter(pl.col("__split_is_test")).drop("__split_is_test")
        train_raw = flagged.filter(~pl.col("__split_is_test")).drop("__split_is_test")
    else:
        n_test = test_size if isinstance(test_size, int) else int(round(n * test_fraction))
        train_raw = shuffled.head(n - n_test)
        test_raw = shuffled.tail(n_test)

    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError(
            f"Split produced an empty side (train={train_raw.height}, "
            f"test={test_raw.height}); adjust test_size."
        )
    return train_raw, test_raw, stratified


def _fill_unfitted_test_nulls(
    test_raw: pl.DataFrame,
    train_raw: pl.DataFrame,
    fitted: Any,
    strategy: str,
) -> tuple[pl.DataFrame, dict[str, Any]]:
    """Fill test gaps the fitted pipeline has no learned value for.

    kailash-ml only stores a fill value for columns that had gaps in the
    rows it was fitted on. A column that is complete in the training rows
    but has gaps in the test rows would otherwise reach the model as null.
    The fill is the TRAINING statistic for the same strategy (never a test
    statistic), and every such column is logged and returned.
    """
    if strategy == "drop":
        return test_raw, {}  # transform() drops incomplete rows itself
    learned = set(fitted.transformers.get("imputer_stats", {}))
    if strategy in ("knn", "iterative"):
        learned |= set(fitted.transformers.get("sklearn_imputer_cols", []))

    fills: dict[str, Any] = {}
    for col in fitted.numeric_columns + fitted.categorical_columns:
        if col in learned or col not in test_raw.columns:
            continue
        if test_raw[col].null_count() == 0:
            continue
        train_col = train_raw[col].drop_nulls()
        if col in fitted.categorical_columns or strategy == "mode":
            modes = train_col.mode().sort()
            if len(modes) == 0:
                raise ValueError(f"Cannot fill '{col}': no observed training values.")
            value = modes[0]
        elif strategy == "median":
            value = train_col.median()
        else:  # mean (and the numeric fallback for any other strategy)
            value = train_col.mean()
        if value is None:
            raise ValueError(f"Cannot fill '{col}': no observed training values.")
        fills[col] = value
    if fills:
        logger.warning(
            "Test rows have gaps in columns that were complete in training; "
            "filled with TRAINING statistics: %s",
            fills,
        )
        test_raw = test_raw.with_columns(
            [pl.col(c).fill_null(v) for c, v in fills.items()]
        )
    return test_raw, fills


def preprocess_train_test(
    train_raw: pl.DataFrame,
    test_raw: pl.DataFrame,
    target: str,
    *,
    seed: int = 42,
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Fit ``PreprocessingPipeline`` on ``train_raw`` only; transform both.

    ``setup_kwargs`` are ``PreprocessingPipeline.setup()`` options
    (``normalize``, ``categorical_encoding``, ``imputation_strategy`` …).
    Steps that change the ROWS of a split are applied to training rows only:

    * ``remove_outliers`` — outlier rows are dropped from training only.
    * ``fix_imbalance`` — SMOTE / ADASYN (or the class-weight flag) is
      applied to the whole transformed training split, never to test.

    Use this directly inside cross-validation folds; use
    :func:`split_then_preprocess` for a single train/test split.
    """
    from kailash_ml import PreprocessingPipeline

    owned = sorted(_OWNED_BY_HELPER & set(setup_kwargs))
    if owned:
        raise TypeError(
            f"{owned} are set by the split-first helper itself and cannot be "
            "passed through to setup()."
        )
    unknown = sorted(set(setup_kwargs) - _SETUP_OPTIONS)
    if unknown:
        raise TypeError(f"Unknown PreprocessingPipeline.setup() options: {unknown}.")
    for name, frame in (("train_raw", train_raw), ("test_raw", test_raw)):
        if target not in frame.columns:
            raise ValueError(f"Target column '{target}' not found in {name}.")
    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError("train_raw and test_raw must both have rows.")
    if set(train_raw.columns) != set(test_raw.columns):
        raise ValueError("train_raw and test_raw must have the same columns.")

    fix_imbalance = bool(setup_kwargs.pop("fix_imbalance", False))
    imbalance_method = setup_kwargs.get("imbalance_method", "smote")
    remove_outliers = bool(setup_kwargs.get("remove_outliers", False))

    pipeline = PreprocessingPipeline()
    # The pipeline only ever sees training rows. Its internal split stays
    # inside them; resampling is applied below to the full training split.
    fitted = pipeline.setup(
        data=train_raw, target=target, seed=seed, fix_imbalance=False, **setup_kwargs
    )

    train_only: list[str] = []
    if remove_outliers:
        # setup() removed the outlier rows (training rows only); its two
        # internal pieces together are every surviving training row.
        train_data = pl.concat([fitted.train_data, fitted.test_data], how="vertical")
        train_only.append(
            f"remove_outliers: {train_raw.height - train_data.height} training rows dropped"
        )
    else:
        train_data = pipeline.transform(train_raw)

    strategy = setup_kwargs.get("imputation_strategy", "mean")
    test_filled, test_fills = _fill_unfitted_test_nulls(test_raw, train_raw, fitted, strategy)
    test_data = pipeline.transform(test_filled)

    if fix_imbalance and fitted.task_type == "classification":
        resample = getattr(pipeline, "_apply_imbalance_correction", None)
        if resample is None:
            raise RuntimeError(
                "This kailash-ml version has no imbalance-correction step on "
                "PreprocessingPipeline; cannot apply fix_imbalance to the training split."
            )
        before = train_data.height
        train_data = resample(train_data, target, imbalance_method, seed)
        train_only.append(
            f"fix_imbalance ({imbalance_method}): training rows {before} -> {train_data.height}"
        )

    summary = "\n".join(
        [
            f"Task type: {fitted.task_type}",
            f"Split first: {train_raw.height} train rows / {test_raw.height} test rows "
            "(pipeline fitted on the train rows only)",
            f"Numeric features: {len(fitted.numeric_columns)}",
            f"Categorical features: {len(fitted.categorical_columns)}",
            f"Encoding: {setup_kwargs.get('categorical_encoding', 'onehot')}",
            f"Normalization: {'yes' if setup_kwargs.get('normalize', True) else 'no'}",
            f"Imputation: {strategy}",
            f"Train-only steps: {'; '.join(train_only) if train_only else 'none'}",
            f"Transformed train: {train_data.height} rows x {train_data.width} cols",
            f"Transformed test: {test_data.height} rows x {test_data.width} cols",
        ]
    )

    return PreprocessResult(
        train_data=train_data,
        test_data=test_data,
        target_column=target,
        task_type=fitted.task_type,
        numeric_columns=list(fitted.numeric_columns),
        categorical_columns=list(fitted.categorical_columns),
        transformers=dict(fitted.transformers),
        pipeline=pipeline,
        original_shape=(train_raw.height + test_raw.height, train_raw.width),
        train_raw_rows=train_raw.height,
        test_raw_rows=test_raw.height,
        stratified=False,
        train_only_steps=tuple(train_only),
        test_null_fills=test_fills,
        summary=summary,
    )


def split_then_preprocess(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Leak-free replacement for ``PreprocessingPipeline().setup(df, ...)``.

    1. Split the RAW frame first (seeded shuffle; stratified on the target
       for classification when ``stratify="auto"``).
    2. Fit ``PreprocessingPipeline`` on the training rows only.
    3. ``transform()`` the test rows with the fitted pipeline.

    ``setup_kwargs`` are passed to ``PreprocessingPipeline.setup()``; see
    :func:`preprocess_train_test` for how row-changing options are handled.
    """
    train_raw, test_raw, stratified = split_raw_train_test(
        df, target, test_size=test_size, seed=seed, stratify=stratify
    )
    result = preprocess_train_test(train_raw, test_raw, target, seed=seed, **setup_kwargs)
    split_line = "stratified on the target" if stratified else "random"
    return replace(
        result,
        stratified=stratified,
        summary=result.summary.replace("Split first:", f"Split first ({split_line}):", 1),
    )


def create_visualizer():
    """Return a ModelVisualizer with the P2 experimental notice acknowledged.

    kailash-ml 2.2.2 emits ExperimentalWarning (a UserWarning) at
    ModelVisualizer construction; exercises run under warnings-as-errors,
    so the notice is acknowledged narrowly here — the ONE construction site
    every course file should use.
    """
    import warnings as _warnings

    from kailash_ml import ModelVisualizer
    from kailash_ml._decorators import ExperimentalWarning as _EW

    with _warnings.catch_warnings():
        _warnings.simplefilter("ignore", _EW)
        return ModelVisualizer()

# ── SDK-internal deprecation nag filter (strict-gate hygiene) ────────────
# torch 2.12's ONNX exporter warns that dynamic_axes is "not recommended"
# under dynamo=True — a deprecation notice inside OnnxBridge's internals,
# not a correctness signal, and fatal under warnings-as-errors. Acknowledged
# here (kailash_helpers is imported by every shared.mlfpNN package, so every
# technique file gets it); message-matched, everything else still errors.
import warnings as _warnings

_warnings.filterwarnings(
    "ignore", message=r".*dynamic_axes.*not recommended.*", category=UserWarning
)
# matplotlib's Agg backend (forced in headless/script runs) nags
# "FigureCanvasAgg is non-interactive, and thus cannot be shown" on every
# plt.show() — a backend notice, not a correctness signal.
_warnings.filterwarnings(
    "ignore", message=r".*non-interactive.*cannot be shown.*", category=UserWarning
)

def hdb_storey_range_expr(column: str = "storey_range"):
    """Polars expression normalising HDB storey_range letter-O typos.

    The raw HDB resale file has letter-O typos ("O4 TO 06", "1O TO 12",
    "28 TO 3O"); read a letter O next to a digit as zero, leaving "TO"
    alone. Single home for the rule (P2): M2 ex_8 and M3 ex_5/07 both
    normalise at the use-site.
    """
    import polars as pl

    return (
        pl.col(column)
        .str.replace_all(r"\bO(\d)", "0${1}")
        .str.replace_all(r"(\d)O\b", "${1}0")
    )



# ── shared/mlfp06/_ollama_bootstrap.py ──
"""Ollama bootstrap for MLFP06 — local LLMs, no API keys, real calls only.

Why this module exists
----------------------
M6 trains, prompts, evaluates, and deploys real LLMs. Routing every call
through OpenAI:

  * forces every learner to hold an API key
  * burns budget every time a notebook reruns a cell
  * makes "did the LLM actually fire?" invisible (a stubbed `cost=$0`
    looks identical to a successful call)

This module switches every M6 LLM call to a locally-running Ollama daemon
that the student installs once and re-uses across every lesson.

Contract
--------
* :func:`make_delegate` — the only sanctioned way to construct a Kaizen
  ``Delegate`` for M6. Wraps the Ollama adapter with sensible defaults and
  forces ``budget_usd=None`` (Kaizen mis-prices Ollama as $3/$15-per-Mtok;
  budget tracking against a free local provider is meaningless).
* :func:`make_embedder` — the only sanctioned way to construct an embedding
  adapter for M6 RAG (lesson 6.4).
* :func:`preflight_ollama` — pre-flight reachability + model-presence check.
  Raises :class:`OllamaUnreachableError` with an actionable message. Call
  this at the top of every notebook / script before constructing a
  Delegate.
* :data:`LESSON_MODELS` — manifest of which models each lesson needs.
  Consumed by ``scripts/generate_selfcontained_notebook.py`` to inject the
  Colab bootstrap cell.

Per ``rules/zero-tolerance.md`` Rule 2: there is no silent stub fallback
in this module. If Ollama is not reachable, we raise. The previous
``run_delegate(...) -> ("unknown", 0.0, latency)`` fallback is BLOCKED.

Per ``rules/framework-first.md``: every LLM call goes through Kaizen
``Delegate``. Raw ``httpx`` calls to ``/api/generate`` are BLOCKED for
generation paths; ``httpx`` is used here only for the daemon-reachability
preflight.
"""

import logging
import os
import shutil
import subprocess
import sys
import time
from typing import Any

import httpx
from dotenv import load_dotenv

logger = logging.getLogger(__name__)

# ════════════════════════════════════════════════════════════════════════
# Configuration — env-driven per rules/env-models.md
# ════════════════════════════════════════════════════════════════════════

load_dotenv()


def _env(key: str, default: str) -> str:
    val = os.environ.get(key)
    return val if val else default


OLLAMA_BASE_URL: str = _env("OLLAMA_BASE_URL", "http://localhost:11434")
"""Where the Ollama daemon listens. Local default; Colab uses the same."""

DEFAULT_CHAT_MODEL: str = _env("OLLAMA_CHAT_MODEL", "llama3.2:3b")
"""Tool-capable instruct model. Allowlisted by Kaizen for function calling."""

DEFAULT_EMBED_MODEL: str = _env("OLLAMA_EMBED_MODEL", "nomic-embed-text")
"""Compact embedding model — ~270MB, 768-dim. Used by M6 RAG (lesson 6.4)."""

DEFAULT_FT_BASE_MODEL: str = _env("OLLAMA_FT_BASE_MODEL", "qwen2.5:0.5b")
"""Tiny base model for fine-tuning demos (lessons 6.2, 6.3). Fits T4."""


# ════════════════════════════════════════════════════════════════════════
# Lesson → models manifest (used by generator + preflight)
# ════════════════════════════════════════════════════════════════════════


LESSON_MODELS: dict[str, list[str]] = {
    # 6.1 Prompting — chat only
    "ex_1": [DEFAULT_CHAT_MODEL],
    # 6.2 Fine-tuning — base for LoRA + chat for evaluation
    "ex_2": [DEFAULT_FT_BASE_MODEL, DEFAULT_CHAT_MODEL],
    # 6.3 Alignment (DPO/GRPO) — base + chat for judge
    "ex_3": [DEFAULT_FT_BASE_MODEL, DEFAULT_CHAT_MODEL],
    # 6.4 RAG — chat + embeddings
    "ex_4": [DEFAULT_CHAT_MODEL, DEFAULT_EMBED_MODEL],
    # 6.5 Agents (ReAct) — tool-capable chat
    "ex_5": [DEFAULT_CHAT_MODEL],
    # 6.6 Multi-agent + MCP — tool-capable chat
    "ex_6": [DEFAULT_CHAT_MODEL],
    # 6.7 PACT governance — chat for governed Delegate
    "ex_7": [DEFAULT_CHAT_MODEL],
    # 6.8 Nexus deployment — chat behind the API
    "ex_8": [DEFAULT_CHAT_MODEL],
}


# ════════════════════════════════════════════════════════════════════════
# Errors
# ════════════════════════════════════════════════════════════════════════


class OllamaUnreachableError(RuntimeError):
    """Daemon not running, daemon unreachable, or required model not pulled.

    The error message MUST be actionable: it tells the user the exact
    command to run to fix the situation. We never wrap a low-level
    ``httpx.ConnectError`` and re-raise without translating it.
    """


# ════════════════════════════════════════════════════════════════════════
# Preflight — reachability + model presence
# ════════════════════════════════════════════════════════════════════════


def _running_in_colab() -> bool:
    return "google.colab" in sys.modules


def _list_pulled_families(*, timeout_s: float = 2.0) -> set[str]:
    """Hit /api/tags and return the set of pulled model families (no tag).

    Returns ``{"llama3.2", "qwen2.5"}`` for a daemon with
    ``llama3.2:3b`` and ``qwen2.5:0.5b`` pulled. The model family is
    everything before the first colon — we match against family rather
    than the full ``family:tag`` because pulling ``llama3.2:3b-instruct``
    or ``llama3.2:3b-instruct-q4_K_M`` should still satisfy a request
    for the ``llama3.2`` family.
    """
    url = f"{OLLAMA_BASE_URL.rstrip('/')}/api/tags"
    try:
        r = httpx.get(url, timeout=timeout_s)
        r.raise_for_status()
    except (httpx.HTTPError, httpx.ConnectError) as exc:
        raise OllamaUnreachableError(
            f"ollama daemon not reachable at {OLLAMA_BASE_URL}. "
            f"Fix: open a terminal and run `ollama serve`. "
            f"On macOS the desktop app starts the daemon automatically. "
            f"Underlying: {type(exc).__name__}: {exc}"
        ) from exc
    payload = r.json() or {}
    families: set[str] = set()
    for entry in payload.get("models", []):
        name = entry.get("name") or entry.get("model") or ""
        if not name:
            continue
        families.add(name.split(":", 1)[0])
    return families


def preflight_ollama(
    *,
    required_models: list[str] | None = None,
    auto_pull: bool | None = None,
    timeout_s: float = 2.0,
) -> None:
    """Verify the daemon is reachable and the required models are pulled.

    Args:
        required_models: Models that must be present, e.g.
            ``["llama3.2:3b", "nomic-embed-text"]``. Family-matched (the
            ``:tag`` portion is informational; any pulled tag for the
            same family satisfies the requirement).
        auto_pull: If ``True``, attempt ``ollama pull <model>`` for any
            missing model via subprocess. Defaults to ``True`` on Colab
            (where the bootstrap cell has just installed the binary)
            and ``False`` locally (where pulling is the student's job
            and surprise downloads are unwelcome).
        timeout_s: HTTP timeout for the reachability check. The default
            is intentionally short — if the daemon is alive it answers
            within 100ms.

    Raises:
        OllamaUnreachableError: Daemon unreachable, or a required model
            is missing after pull (or auto_pull was disabled).
    """
    if auto_pull is None:
        auto_pull = _running_in_colab()

    have = _list_pulled_families(timeout_s=timeout_s)
    needed = required_models or []
    missing = [m for m in needed if m.split(":", 1)[0] not in have]

    if missing and auto_pull:
        for model in missing:
            _pull_model(model)
        # Re-check
        have = _list_pulled_families(timeout_s=timeout_s)
        missing = [m for m in needed if m.split(":", 1)[0] not in have]

    if missing:
        cmds = "\n  ".join(f"ollama pull {m}" for m in missing)
        raise OllamaUnreachableError(
            f"Required model(s) not pulled: {missing}.\n"
            f"Fix — run in a terminal:\n  {cmds}"
        )

    logger.info(
        "ollama.preflight.ok",
        extra={
            "base_url": OLLAMA_BASE_URL,
            "models_present": sorted(have),
            "required": needed,
        },
    )


def _pull_model(model: str) -> None:
    """Run ``ollama pull <model>`` as a subprocess. Streams to stdout."""
    if shutil.which("ollama") is None:
        raise OllamaUnreachableError(
            f"`ollama` binary not on PATH — cannot pull {model}. "
            f"Install: https://ollama.com/download"
        )
    print(f"[ollama] pulling {model} (one-time download) …", flush=True)
    t0 = time.monotonic()
    proc = subprocess.run(
        ["ollama", "pull", model],
        capture_output=False,
        text=True,
    )
    elapsed = time.monotonic() - t0
    if proc.returncode != 0:
        raise OllamaUnreachableError(
            f"`ollama pull {model}` failed with exit {proc.returncode} "
            f"after {elapsed:.1f}s. Check network and disk space."
        )
    print(f"[ollama] {model} pulled in {elapsed:.1f}s", flush=True)


# ════════════════════════════════════════════════════════════════════════
# Delegate factory — the only sanctioned constructor for M6
# ════════════════════════════════════════════════════════════════════════


def make_delegate(
    *,
    model: str | None = None,
    tools: Any = None,
    system_prompt: str | None = None,
    temperature: float = 0.4,
    max_tokens: int = 4096,
    **kwargs: Any,
) -> Any:
    """Construct a Kaizen ``Delegate`` backed by the local Ollama daemon.

    This is the ONLY way M6 helpers and exercises construct a Delegate.
    Direct ``Delegate(...)`` construction in M6 code is BLOCKED — the
    cost-mispricing trap (``budget_usd`` against a free provider) and
    the silent OpenAI fallback that comes from omitting ``adapter=`` are
    the reasons.

    Args:
        model: Ollama model tag (e.g. ``"llama3.2:3b"``). Defaults to
            :data:`DEFAULT_CHAT_MODEL`.
        tools: Tool registry or list — passed through to Delegate.
            Auto-disables streaming on the Ollama adapter.
        system_prompt: Optional system prompt.
        temperature: Sampling temperature. Default 0.4.
        max_tokens: Hard cap on output length. Default 4096.
        **kwargs: Forwarded to ``Delegate(...)``. ``budget_usd`` is
            forced to ``None`` and ``api_key`` / ``base_url`` are
            BLOCKED (the adapter owns those).

    Returns:
        A configured Kaizen ``Delegate`` instance.
    """
    for blocked in ("api_key", "base_url", "budget_usd", "adapter"):
        if blocked in kwargs:
            raise ValueError(
                f"make_delegate(): {blocked!r} is owned by the bootstrap "
                f"and cannot be overridden by callers."
            )

    # Lazy imports — keep this module importable in environments that have
    # not yet installed kaizen-agents (e.g. early bootstrap cells).
    from kaizen_agents import Delegate
    from kaizen_agents.delegate.adapters.registry import get_adapter

    resolved = model or DEFAULT_CHAT_MODEL
    adapter = get_adapter(
        "ollama",
        model=resolved,
        base_url=OLLAMA_BASE_URL,
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return Delegate(
        model=resolved,
        adapter=adapter,
        tools=tools,
        system_prompt=system_prompt,
        budget_usd=None,
        **kwargs,
    )


# ════════════════════════════════════════════════════════════════════════
# Embedding factory — the only sanctioned constructor for M6 RAG
# ════════════════════════════════════════════════════════════════════════


def make_embedder(*, model: str | None = None) -> Any:
    """Construct a Kaizen embedding adapter backed by Ollama.

    Args:
        model: Ollama embedding model tag. Defaults to
            :data:`DEFAULT_EMBED_MODEL`.

    Returns:
        An adapter with ``async def embed(texts: list[str]) -> list[list[float]]``.
    """
    from kaizen_agents.delegate.adapters.registry import get_embedding_adapter

    resolved = model or DEFAULT_EMBED_MODEL
    return get_embedding_adapter(
        "ollama",
        model=resolved,
        base_url=OLLAMA_BASE_URL,
    )


# ════════════════════════════════════════════════════════════════════════
# Streaming helpers — same call shape as the old `run_delegate`
# ════════════════════════════════════════════════════════════════════════


async def run_delegate_text(
    delegate: Any, prompt: str
) -> tuple[str, dict[str, int], float]:
    """Run a Delegate to completion, return ``(text, usage, latency_s)``.

    ``usage`` is a dict with ``prompt_tokens``, ``completion_tokens``,
    ``total_tokens`` (zero for events that don't carry usage). ``cost``
    is intentionally absent — Ollama is free, and Kaizen's ``consumed_usd``
    is mis-priced for local models.

    Event handling: the Kaizen ``TextDelta`` events expose the streamed
    chunk on ``.text`` (not ``.delta_text`` — that name is reserved for a
    different StreamEvent shape). The terminal ``TurnComplete`` event
    carries the final ``.text`` (entire response, not a delta) plus the
    ``.usage`` dict. We accumulate from ``TextDelta.text`` for streaming
    UX and fall back to ``TurnComplete.text`` if no deltas arrived.
    """
    t0 = time.perf_counter()
    response = ""
    usage: dict[str, int] = {
        "prompt_tokens": 0,
        "completion_tokens": 0,
        "total_tokens": 0,
    }
    final_text: str | None = None
    async for event in delegate.run(prompt):
        ev_type = getattr(event, "event_type", None)
        text = getattr(event, "text", None)
        if ev_type == "text_delta" and text:
            response += text
        elif ev_type == "turn_complete":
            final_text = text or final_text
            ev_usage = getattr(event, "usage", None) or {}
            for k in usage:
                if k in ev_usage:
                    usage[k] = int(ev_usage[k])
    if not response and final_text:
        response = final_text
    elapsed = time.perf_counter() - t0
    return response, usage, elapsed


# ════════════════════════════════════════════════════════════════════════
# Colab installer — used by the generator's Cell 0
# ════════════════════════════════════════════════════════════════════════


def colab_install_and_start_ollama(*, log_path: str = "/tmp/ollama.log") -> None:
    """Install the Ollama binary and start the daemon (Colab only).

    Idempotent: skips install if the binary is already on PATH and
    skips daemon start if a process is already listening on
    :data:`OLLAMA_BASE_URL`. Raises :class:`OllamaUnreachableError`
    if either step fails.

    Not for local use — locally, students run ``ollama serve`` themselves.
    """
    if not _running_in_colab():
        raise RuntimeError(
            "colab_install_and_start_ollama() is Colab-only. "
            "On a local machine, run `ollama serve` in a terminal."
        )

    # Step 1 — install if missing
    if shutil.which("ollama") is None:
        print("[colab] installing ollama …", flush=True)
        proc = subprocess.run(
            "curl -fsSL https://ollama.com/install.sh | sh",
            shell=True,
            check=False,
            capture_output=True,
            text=True,
        )
        if proc.returncode != 0:
            raise OllamaUnreachableError(
                f"ollama install failed (exit {proc.returncode}):\n"
                f"stdout: {proc.stdout[-500:]}\nstderr: {proc.stderr[-500:]}"
            )

    # Step 2 — start daemon if not already listening
    try:
        httpx.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=1.0).raise_for_status()
        print("[colab] ollama daemon already running", flush=True)
        return
    except (httpx.HTTPError, httpx.ConnectError):
        pass

    print(f"[colab] starting ollama daemon (logs → {log_path}) …", flush=True)
    subprocess.Popen(
        f"nohup ollama serve > {log_path} 2>&1 &",
        shell=True,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )

    # Step 3 — wait up to 30s for the daemon to answer /api/tags
    deadline = time.monotonic() + 30.0
    last_err: Exception | None = None
    while time.monotonic() < deadline:
        try:
            httpx.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=1.0).raise_for_status()
            print("[colab] ollama daemon ready", flush=True)
            return
        except (httpx.HTTPError, httpx.ConnectError) as exc:
            last_err = exc
            time.sleep(0.5)

    raise OllamaUnreachableError(
        f"ollama daemon failed to come up within 30s. "
        f"Last error: {type(last_err).__name__}: {last_err}. "
        f"Inspect {log_path} for details."
    )


# ── shared/mlfp06/ex_2.py ──
"""
Shared infrastructure for Exercise 2 — LLM Fine-Tuning.

Contains: SFT dataset loader (IMDB -> instruction pairs), LoRA/adapter
parameter-counting helpers, kailash-align config factory, base
hyperparameters, and output directory. Technique-specific classes
(LoRALayer, AdapterLayer, etc.) live in the per-technique files.
"""

import os
from pathlib import Path

import polars as pl
import torch


# ════════════════════════════════════════════════════════════════════════
# ENVIRONMENT SETUP
# ════════════════════════════════════════════════════════════════════════

setup_environment()
torch.manual_seed(42)
device = get_device()

# Output directory for all visualisation and training artifacts
REPO_ROOT = Path.cwd()
OUTPUT_DIR = REPO_ROOT / "outputs" / "ex2_finetuning"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ════════════════════════════════════════════════════════════════════════
# SHARED HYPERPARAMETERS
# ════════════════════════════════════════════════════════════════════════

D_MODEL = 512  # synthetic demo hidden dim for from-scratch LoRA / adapter
LORA_RANK = 8  # default LoRA rank for the demos
LORA_ALPHA = 16.0  # typical alpha = 2 * rank
ADAPTER_BOTTLENECK = 64  # default adapter bottleneck

# ════════════════════════════════════════════════════════════════════════
# SFT DATA LOADING — IMDB instruction-response pairs
# ════════════════════════════════════════════════════════════════════════

DATA_DIR = REPO_ROOT / "data" / "mlfp06" / "imdb"
DATA_DIR.mkdir(parents=True, exist_ok=True)
CACHE_FILE = DATA_DIR / "imdb_sft_2k.parquet"


def load_imdb_sft(
    n_rows: int = 2000, train_frac: float = 0.9
) -> tuple[pl.DataFrame, pl.DataFrame, pl.DataFrame]:
    """Load IMDB movie reviews and reformat as SFT instruction pairs.

    Returns:
        (full, train, eval) polars DataFrames with columns:
            instruction, response, text, label
    """
    if CACHE_FILE.exists():
        print(f"Loading cached IMDB SFT pairs from {CACHE_FILE}")
        sft_data = pl.read_parquet(CACHE_FILE)
    else:
        print("Downloading stanfordnlp/imdb from HuggingFace (first run)...")
        from datasets import load_dataset

        ds = load_dataset("stanfordnlp/imdb", split="train")
        ds = ds.shuffle(seed=42).select(range(min(n_rows, len(ds))))

        label_names = {0: "negative", 1: "positive"}
        rows = []
        for row in ds:
            review = row["text"][:1500]
            sentiment = label_names[row["label"]]
            rows.append(
                {
                    "instruction": (
                        "Classify the sentiment of the following movie review as "
                        "either 'positive' or 'negative', then briefly justify "
                        f"your answer.\n\nReview: {review}"
                    ),
                    "response": (
                        f"Sentiment: {sentiment}. The reviewer expresses a clearly "
                        f"{sentiment} reaction to the film."
                    ),
                    "text": review,
                    "label": sentiment,
                }
            )
        sft_data = pl.DataFrame(rows)
        sft_data.write_parquet(CACHE_FILE)
        print(f"Cached {sft_data.height} SFT pairs to {CACHE_FILE}")

    n_train = int(sft_data.height * train_frac)
    train_data = sft_data[:n_train]
    eval_data = sft_data[n_train:]
    print(
        f"IMDB SFT loaded: {sft_data.height} pairs "
        f"({train_data.height} train / {eval_data.height} eval)"
    )
    return sft_data, train_data, eval_data


# ════════════════════════════════════════════════════════════════════════
# PARAMETER COUNTING HELPERS
# ════════════════════════════════════════════════════════════════════════


def count_lora_params(
    d_model: int, rank: int, num_modules: int = 1, num_layers: int = 1
) -> int:
    """LoRA trainable params = 2 * d * r per target module per layer.

    For d=512, r=8, modules=1, layers=1 -> 8,192 params.
    """
    return (d_model * rank + rank * d_model) * num_modules * num_layers


def count_adapter_params(d_model: int, bottleneck: int, num_layers: int = 1) -> int:
    """Adapter params per layer: down(d*b+b) + up(b*d+d) + layernorm(2*d).

    Matches the AdapterLayer in 02_adapter_from_scratch.py.
    """
    return (
        d_model * bottleneck + bottleneck + bottleneck * d_model + d_model + 2 * d_model
    ) * num_layers


def full_finetune_params(
    d_model: int, num_modules: int = 1, num_layers: int = 1
) -> int:
    """Full fine-tuning baseline: d^2 per module per layer."""
    return d_model * d_model * num_modules * num_layers


# ════════════════════════════════════════════════════════════════════════
# KAILASH-ALIGN CONFIG FACTORY
# ════════════════════════════════════════════════════════════════════════


def get_base_model_name() -> str:
    """Return the HuggingFace repo id of the SFT base model.

    Resolution: ``SFT_BASE_MODEL`` env var → ``Qwen/Qwen2.5-0.5B-Instruct``.
    Note: Align/TRL training needs a HuggingFace repo id, NOT an Ollama tag —
    the Ollama-side ``OLLAMA_FT_BASE_MODEL`` is for inference of the
    fine-tuned model after GGUF export, not for training.
    """
    return os.environ.get("SFT_BASE_MODEL") or "Qwen/Qwen2.5-0.5B-Instruct"


def build_sft_config(
    base_model: str | None = None,
    lora_r: int = 16,
    lora_alpha: int = 32,
    num_epochs: int = 3,
    output_subdir: str = "sft_output",
):
    """Build a kailash-align AlignmentConfig for SFT+LoRA training.

    Imported lazily so technique files that do not call SFT are not
    forced to install the align extra at import time.

    kailash-align 0.6.0+ uses a composed AlignmentConfig: top-level
    method + base_model_id + experiment_dir, plus required LoRAConfig
    + SFTConfig + DPOConfig sub-configs. The DPOConfig is unused for
    SFT but the constructor still expects it (default values are fine).
    """
    from kailash_align import AlignmentConfig, DPOConfig, LoRAConfig, SFTConfig

    return AlignmentConfig(
        method="sft",
        base_model_id=base_model or get_base_model_name(),
        lora=LoRAConfig(
            rank=lora_r,
            alpha=lora_alpha,
            target_modules=("q_proj", "v_proj"),
            dropout=0.05,
        ),
        sft=SFTConfig(
            num_train_epochs=num_epochs,
            per_device_train_batch_size=4,
            gradient_accumulation_steps=4,
            learning_rate=2e-4,
            warmup_ratio=0.1,
            max_seq_length=512,
        ),
        dpo=DPOConfig(),
        experiment_dir=str(OUTPUT_DIR / output_subdir),
    )


# ── shared/mlfp06/ex_3.py ──
"""
Shared infrastructure for MLFP06 Exercise 3 — DPO Preference Alignment.

Contains: UltraFeedback loading, DPO loss helper, GRPO advantage helper,
visualisation utilities, output paths. Technique-specific code does NOT
belong here.
"""

import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import torch
import torch.nn.functional as F


# ════════════════════════════════════════════════════════════════════════
# ENVIRONMENT SETUP
# ════════════════════════════════════════════════════════════════════════

setup_environment()
torch.manual_seed(42)
np.random.seed(42)
device = get_device()


MODEL_NAME = DEFAULT_CHAT_MODEL
# Single home for the documented default: shared.mlfp06.ex_2.get_base_model_name
BASE_MODEL = get_base_model_name()

# Output directories
OUTPUT_DIR = Path("outputs") / "ex3_dpo"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATA_CACHE = Path("data") / "mlfp06" / "ultrafeedback"
DATA_CACHE.mkdir(parents=True, exist_ok=True)
CACHE_FILE = DATA_CACHE / "ultrafeedback_2k.parquet"

ADAPTER_OUTPUT_DIR = OUTPUT_DIR / "dpo_output"


# ════════════════════════════════════════════════════════════════════════
# DATA LOADING — UltraFeedback Binarized preference pairs
# ════════════════════════════════════════════════════════════════════════


def load_ultrafeedback(n_samples: int = 2000) -> pl.DataFrame:
    """Load UltraFeedback Binarized preference dataset as a polars DataFrame.

    Columns: prompt, chosen, rejected. Cached as parquet after first load.
    """
    if CACHE_FILE.exists():
        print(f"Loading cached preference pairs from {CACHE_FILE}")
        return pl.read_parquet(CACHE_FILE)

    print("Downloading UltraFeedback Binarized from HuggingFace (first run)...")
    from datasets import load_dataset

    ds = load_dataset("trl-lib/ultrafeedback_binarized", split="train")
    ds = ds.shuffle(seed=42).select(range(min(n_samples, len(ds))))

    def _extract(row: dict) -> dict:
        chosen_msgs = row["chosen"]
        rejected_msgs = row["rejected"]
        prompt = ""
        for msg in chosen_msgs:
            if msg.get("role") == "user":
                prompt = msg.get("content", "")
                break
        chosen_text = next(
            (m["content"] for m in chosen_msgs if m.get("role") == "assistant"),
            "",
        )
        rejected_text = next(
            (m["content"] for m in rejected_msgs if m.get("role") == "assistant"),
            "",
        )
        return {
            "prompt": prompt,
            "chosen": chosen_text,
            "rejected": rejected_text,
        }

    rows = [_extract(r) for r in ds]
    rows = [r for r in rows if r["prompt"] and r["chosen"] and r["rejected"]]
    pref_data = pl.DataFrame(rows)
    pref_data.write_parquet(CACHE_FILE)
    print(f"Cached {pref_data.height} preference pairs to {CACHE_FILE}")
    return pref_data


def split_preferences(
    pref_data: pl.DataFrame, train_frac: float = 0.9
) -> tuple[pl.DataFrame, pl.DataFrame]:
    """Split preference pairs into train/eval."""
    n_train = int(pref_data.height * train_frac)
    return pref_data[:n_train], pref_data[n_train:]


# ════════════════════════════════════════════════════════════════════════
# DPO LOSS — canonical implementation (re-used across technique files)
# ════════════════════════════════════════════════════════════════════════


def dpo_loss(
    policy_chosen_logps: torch.Tensor,
    policy_rejected_logps: torch.Tensor,
    ref_chosen_logps: torch.Tensor,
    ref_rejected_logps: torch.Tensor,
    beta: float = 0.1,
) -> torch.Tensor:
    """Compute the DPO loss from scratch.

    Args:
        policy_chosen_logps:   log P_policy(y_w | x)   [batch]
        policy_rejected_logps: log P_policy(y_l | x)   [batch]
        ref_chosen_logps:      log P_ref(y_w | x)      [batch]
        ref_rejected_logps:    log P_ref(y_l | x)      [batch]
        beta:                  Temperature controlling preference strength.

    Returns:
        Scalar DPO loss (averaged over batch).
    """
    chosen_log_ratio = policy_chosen_logps - ref_chosen_logps
    rejected_log_ratio = policy_rejected_logps - ref_rejected_logps
    logits = beta * (chosen_log_ratio - rejected_log_ratio)
    return -F.logsigmoid(logits).mean()


# ════════════════════════════════════════════════════════════════════════
# GRPO — group-relative advantage helper
# ════════════════════════════════════════════════════════════════════════


def grpo_advantages(rewards: torch.Tensor, eps: float = 1e-6) -> torch.Tensor:
    """Compute GRPO group-relative advantages (Shao et al., 2024, DeepSeekMath).

        A_i = (r_i - mean(r_group)) / (std(r_group) + eps)

    rewards: [n_prompts, K] tensor of scalar rewards, K completions per prompt.
    Returns advantages of the same shape: each row has mean ~0 and std ~1,
    so the update size does not depend on the reward scale. A group whose
    completions all score the same gets advantage 0 (no learning signal).
    """
    group_mean = rewards.mean(dim=1, keepdim=True)
    group_std = rewards.std(dim=1, keepdim=True)
    return (rewards - group_mean) / (group_std + eps)


# ════════════════════════════════════════════════════════════════════════
# VISUALISATION — "Seeing Is Believing" for alignment
# ════════════════════════════════════════════════════════════════════════


def show_preference_length_distribution(
    pref_data: pl.DataFrame, title: str = "Chosen vs Rejected Length"
) -> Path:
    """Histogram of chosen vs rejected response lengths (character count)."""
    chosen_lens = np.array([len(s) for s in pref_data["chosen"].to_list()])
    rejected_lens = np.array([len(s) for s in pref_data["rejected"].to_list()])

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.hist(
        chosen_lens,
        bins=60,
        alpha=0.65,
        label=f"Chosen (median={int(np.median(chosen_lens))})",
        color="#2E7D32",
    )
    ax.hist(
        rejected_lens,
        bins=60,
        alpha=0.65,
        label=f"Rejected (median={int(np.median(rejected_lens))})",
        color="#C62828",
    )
    ax.set_xlabel("Response length (characters)")
    ax.set_ylabel("Count")
    ax.set_title(title, fontsize=13, fontweight="bold")
    ax.legend()
    fname = OUTPUT_DIR / "ex3_preference_lengths.png"
    plt.tight_layout()
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fname}")
    return fname


def show_dpo_loss_curve_by_margin(
    beta: float = 0.1, title: str = "DPO loss vs policy preference margin"
) -> Path:
    """Visual proof: as the policy's preference margin grows, DPO loss drops."""
    margins = torch.linspace(-3, 3, 121)
    losses = [-F.logsigmoid(torch.tensor(beta * m)).item() for m in margins]

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(margins.numpy(), losses, color="#1565C0", linewidth=2.2)
    ax.axvline(0, color="#616161", linestyle=":", label="No preference (margin=0)")
    ax.set_xlabel("Policy log-ratio margin  (chosen - rejected)")
    ax.set_ylabel("DPO loss")
    ax.set_title(f"{title}  (beta={beta})", fontsize=13, fontweight="bold")
    ax.legend()
    ax.grid(True, alpha=0.3)
    fname = OUTPUT_DIR / "ex3_dpo_loss_curve.png"
    plt.tight_layout()
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fname}")
    return fname


def show_beta_sensitivity(betas: list[float], losses: list[float]) -> Path:
    """Bar chart of DPO loss for a synthetic batch across beta values."""
    fig, ax = plt.subplots(figsize=(9, 5))
    colors = [
        "#90CAF9" if b <= 0.1 else "#FFB74D" if b <= 0.3 else "#E57373" for b in betas
    ]
    bars = ax.bar([f"{b}" for b in betas], losses, color=colors, edgecolor="black")
    for bar, loss_val in zip(bars, losses):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.005,
            f"{loss_val:.3f}",
            ha="center",
            fontsize=9,
        )
    ax.set_xlabel("Beta (KL-penalty coefficient)")
    ax.set_ylabel("DPO loss (synthetic batch)")
    ax.set_title(
        "Beta sensitivity — same margin, higher beta = lower loss (less drift needed)",
        fontsize=13,
        fontweight="bold",
    )
    fname = OUTPUT_DIR / "ex3_beta_sensitivity.png"
    plt.tight_layout()
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fname}")
    return fname


def show_grpo_advantages(rewards: torch.Tensor, advantages: torch.Tensor) -> Path:
    """Visualise GRPO advantages for each prompt group."""
    n_prompts, K = rewards.shape
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    im0 = axes[0].imshow(rewards.numpy(), aspect="auto", cmap="viridis")
    axes[0].set_title("Rewards  r(x, y_i)", fontsize=12, fontweight="bold")
    axes[0].set_xlabel(f"Completion index (K={K})")
    axes[0].set_ylabel("Prompt index")
    fig.colorbar(im0, ax=axes[0])

    im1 = axes[1].imshow(
        advantages.numpy(),
        aspect="auto",
        cmap="RdBu_r",
        vmin=-abs(advantages).max(),
        vmax=abs(advantages).max(),
    )
    axes[1].set_title(
        "GRPO advantages  (r_i - mean) / std", fontsize=12, fontweight="bold"
    )
    axes[1].set_xlabel(f"Completion index (K={K})")
    axes[1].set_ylabel("Prompt index")
    fig.colorbar(im1, ax=axes[1])

    fname = OUTPUT_DIR / "ex3_grpo_advantages.png"
    plt.tight_layout()
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fname}")
    return fname


def show_safety_refusal_rates(
    base_rate: float, aligned_rate: float, n_prompts: int
) -> Path:
    """Bar chart comparing base-model vs DPO-aligned refusal rates."""
    fig, ax = plt.subplots(figsize=(8, 5))
    labels = ["Base (pre-DPO)", "DPO-aligned"]
    values = [base_rate * 100, aligned_rate * 100]
    colors = ["#9E9E9E", "#2E7D32"]
    bars = ax.bar(labels, values, color=colors, edgecolor="black")
    for bar, val in zip(bars, values):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 1,
            f"{val:.0f}%",
            ha="center",
            fontsize=12,
            fontweight="bold",
        )
    ax.set_ylim(0, 110)
    ax.set_ylabel("Refusal rate on adversarial prompts (%)")
    ax.set_title(
        f"DPO safety alignment — {n_prompts} adversarial prompts",
        fontsize=13,
        fontweight="bold",
    )
    fname = OUTPUT_DIR / "ex3_safety_refusal_rates.png"
    plt.tight_layout()
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fname}")
    return fname




# ════════════════════════════════════════════════════════════════════════
# MLFP06 — Exercise 3.4: GRPO and LLM-as-Judge Evaluation
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Understand GRPO (Group Relative Policy Optimization) and when it beats DPO
#   - Compute std-normalised group-relative advantages and the clipped GRPO objective
#   - Visualise GRPO advantages as a reward heatmap
#   - Run an LLM-as-judge evaluation with Kaizen Delegate
#   - Measure two known biases (position, verbosity) and mitigate position
#     bias with a swap-averaged judge
#   - Survey standard benchmarks (MMLU, HellaSwag, HumanEval, MT-Bench, etc.)
#   - Apply to a Singapore public hospital's clinical RAG assistant
#
# PREREQUISITES: 03_dpo_training.py (you trained a DPO adapter).
# ESTIMATED TIME: ~50 min
#
# TASKS:
#   1. GRPO theory: group-relative advantages + clipped objective with KL
#   2. Visualise GRPO advantages
#   3. LLM-as-judge: compare two responses with Kaizen Delegate
#   4. Position bias test (swap A/B) and a swap-averaged judge
#   5. Verbosity bias test (concise vs padded response)
#   6. Benchmarks survey (MMLU, HellaSwag, HumanEval, MT-Bench, etc.)
#   7. Apply: hospital clinical RAG assistant evaluation plan
#
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import json

import polars as pl
import torch

# Delegate construction routes through shared.mlfp06._ollama_bootstrap.



## THEORY — GRPO vs DPO in one page


All inputs are [n_prompts, K]. Returns (loss, mean_kl). The KL uses the
    unbiased estimator from DeepSeekMath: pi_ref/pi - log(pi_ref/pi) - 1.


In [ ]:
# GRPO (Group Relative Policy Optimization) was introduced by Shao et al.
# (2024) in DeepSeekMath and later used to train DeepSeek-R1 (2025).
#
#   For each prompt x, sample K completions from the current policy:
#     y_1, y_2, ..., y_K ~ pi_old(y|x)
#
#   Score each completion with a reward function r(x, y_i).
#
#   Advantage = reward standardised WITHIN the group (no value network):
#     A_i = (r_i - mean(r_1..r_K)) / (std(r_1..r_K) + eps)
#
#   Objective = PPO-style clipped surrogate + KL penalty to the reference:
#     rho_i  = pi_theta(y_i|x) / pi_old(y_i|x)
#     L_GRPO = -E[ (1/K) sum_i min(rho_i * A_i,
#                                  clip(rho_i, 1-eps_c, 1+eps_c) * A_i) ]
#              + beta_kl * KL(pi_theta || pi_ref)
#   (The real objective averages per token; this exercise works per
#    completion to keep the arithmetic visible.)
#
# DPO vs GRPO:
#   DPO:   pairwise preferences (chosen vs rejected)
#          Best when: preference pairs are available
#          Simpler:   closed-form loss, no sampling
#   GRPO:  group-relative scoring over K samples
#          Best when: a verifiable reward function exists
#                     (math correctness, code execution, unit tests)
#          Flexible:  any reward function, not just pairwise
#
# Why the std division matters: subtracting the mean alone keeps the
# reward's units — multiply every reward by 10 and every update is 10x
# larger. Dividing by the group std makes the advantages scale-free, so
# only the RELATIVE quality inside the group drives the update. A group
# where every completion scores the same carries no signal (A_i = 0).

print("=" * 70)
print("TASK 1: GRPO — Group Relative Policy Optimization")
print("=" * 70)

torch.manual_seed(42)
K = 5
n_prompts = 8
rewards = torch.randn(n_prompts, K)
# TODO: Compute the GRPO advantages for `rewards`, and again for
#       `rewards * 10.0` (to prove they are scale-free).
#       Hint: the shared ex_3 module has a group-relative advantage helper;
#       standardise within each prompt's row (dim=1).
advantages = ____
advantages_scaled = ____

print(f"  Prompts: {n_prompts}, Completions per prompt: K={K}")
print(f"  Rewards (prompt 0): {[round(r, 3) for r in rewards[0].tolist()]}")
print(f"  Group mean / std:   {rewards[0].mean().item():.4f} / {rewards[0].std().item():.4f}")
print(f"  Advantages (prompt 0): {[round(a, 3) for a in advantages[0].tolist()]}")
print(f"  Advantage sum per group ~ 0: {advantages.sum(dim=1).mean().item():.6f}")
print(f"  Advantage std per group ~ 1: {advantages.std(dim=1).mean().item():.4f}")
scale_gap = (advantages - advantages_scaled).abs().max().item()
print(f"  Max change when rewards are x10: {scale_gap:.2e}  (scale-free)")


def grpo_objective(
    logp_new: torch.Tensor,
    logp_old: torch.Tensor,
    logp_ref: torch.Tensor,
    adv: torch.Tensor,
    clip_eps: float = 0.2,
    kl_beta: float = 0.04,
) -> tuple[torch.Tensor, torch.Tensor]:
    # TODO: probability ratio pi_theta / pi_old from the log-probs
    ratio = ____
    # TODO: clipped surrogate — elementwise min of the unclipped and the
    #       clipped (torch.clamp to [1-clip_eps, 1+clip_eps]) terms, then mean
    surrogate = ____
    # TODO: KL estimator with log_ref_ratio = logp_ref - logp_new:
    #       exp(log_ref_ratio) - log_ref_ratio - 1, averaged
    log_ref_ratio = logp_ref - logp_new
    kl = ____
    return -surrogate + kl_beta * kl, kl


# Sanity case: before any update, pi_theta == pi_old == pi_ref.
logp_old = torch.randn(n_prompts, K) - 5.0
loss_start, kl_start = grpo_objective(logp_old, logp_old, logp_old, advantages)
# After a step that raises the log-prob of high-advantage completions:
logp_new = logp_old + 0.5 * advantages
loss_step, kl_step = grpo_objective(logp_new, logp_old, logp_old, advantages)
# A huge step is clipped — the surrogate cannot keep improving.
logp_big = logp_old + 5.0 * advantages
loss_big, kl_big = grpo_objective(logp_big, logp_old, logp_old, advantages)
print(f"\n  GRPO loss at start (ratio=1, KL=0): {loss_start.item():+.4f}  KL={kl_start.item():.4f}")
print(f"  GRPO loss after a small step:       {loss_step.item():+.4f}  KL={kl_step.item():.4f}")
print(f"  GRPO loss after a huge step:        {loss_big.item():+.4f}  KL={kl_big.item():.4f}")
print(
    "  A small step towards high-advantage completions lowers the loss. A huge\n"
    "  step gains nothing from the clipped surrogate and pays a large KL\n"
    "  penalty — which is exactly what keeps GRPO updates conservative."
)



### Checkpoint 1


In [ ]:
assert advantages.shape == rewards.shape
assert abs(advantages.sum(dim=1).mean().item()) < 1e-5
assert abs(advantages.std(dim=1).mean().item() - 1.0) < 1e-3, "advantages must be std-normalised"
assert scale_gap < 1e-4, "GRPO advantages must not depend on the reward scale"
assert abs(loss_start.item()) < 1e-5 and kl_start.item() < 1e-6
assert loss_step.item() < loss_start.item(), "a step towards high-advantage completions lowers the loss"
assert loss_big.item() > loss_step.item(), "clip + KL must penalise an oversized step"
print("✓ Checkpoint 1 passed — GRPO advantages and objective verified\n")



## VISUALISE — Reward and advantage heatmaps


In [ ]:
show_grpo_advantages(rewards, advantages)
assert (OUTPUT_DIR / "ex3_grpo_advantages.png").exists()
print("✓ Visual checkpoint passed — GRPO advantage heatmap saved\n")



## TASK 3 — LLM-as-judge evaluation


The judge replied, but not with a usable JSON verdict.


Raises JudgeParseError when the reply is not a valid verdict — the
    caller decides how to count it. It is NEVER silently turned into a tie.


Query: {prompt[:500]}

Response A:
{response_a[:500]}

Response B:
{response_b[:500]}

Evaluate on: helpfulness, accuracy, clarity, safety.
Output ONLY a JSON object:


Each response is scored once in slot A and once in slot B, so a judge
    that favours one slot adds the same bonus to both — it cancels out.


In [ ]:
print("=" * 70)
print("TASK 3: LLM-as-judge — compare two responses with Kaizen Delegate")
print("=" * 70)

# Fail loudly and early if the local LLM is not available. There is no
# offline fallback: a judge that cannot run must not produce verdicts.
preflight_ollama(required_models=[MODEL_NAME])


class JudgeParseError(ValueError):


async def llm_judge(prompt: str, response_a: str, response_b: str) -> dict:
    # make_delegate() reads the model from OLLAMA_CHAT_MODEL and backs the
    # call with the local Ollama daemon (no API keys, no OpenAI fallback).
    # TODO: Build the judge Delegate with the course bootstrap factory.
    #       Do NOT pass model= (the model comes from OLLAMA_CHAT_MODEL);
    #       use a deterministic temperature for judging.
    delegate = ____

    response, *_ = await run_delegate_text(delegate, judge_prompt)

    try:
        start = response.index("{")
        end = response.rindex("}") + 1
        verdict = json.loads(response[start:end])
        verdict["score_a"] = float(verdict["score_a"])
        verdict["score_b"] = float(verdict["score_b"])
    except (ValueError, KeyError, TypeError) as exc:
        raise JudgeParseError(f"unparseable verdict: {response[:200]!r}") from exc
    if verdict.get("winner") not in ("A", "B", "tie"):
        raise JudgeParseError(f"invalid winner {verdict.get('winner')!r}")
    return verdict


async def swap_averaged_judge(prompt: str, x: str, y: str) -> dict:
    xy = await llm_judge(prompt, x, y)
    yx = await llm_judge(prompt, y, x)
    # TODO: average each response's score across BOTH slots
    #       (x was slot A in xy and slot B in yx; y the reverse).
    score_x = ____
    score_y = ____
    winner = "x" if score_x > score_y else "y" if score_y > score_x else "tie"
    return {"winner": winner, "score_x": score_x, "score_y": score_y}



## TASK 4 — Position bias test (swap A/B) and its mitigation


A pair is CONSISTENT only when the judge picks the good answer in BOTH
    orders. Unparseable verdicts are counted as failures, not as agreement.


How often does the swap-averaged judge pick the good answer?


In [ ]:
JUDGE_PAIRS = [
    {
        "prompt": "What is the capital of Singapore?",
        "good": "Singapore is a city-state; the entire country is the capital.",
        "bad": "Probably somewhere in Asia. I don't remember exactly.",
    },
    {
        "prompt": "How does public key cryptography work?",
        "good": "Each party has a public key (shared) and a private key (secret). "
        "Messages encrypted with the public key can only be decrypted by "
        "the matching private key.",
        "bad": "It uses two keys somehow. One is public.",
    },
    {
        "prompt": "Summarise photosynthesis in one sentence.",
        "good": "Plants convert sunlight, water, and CO2 into glucose and oxygen "
        "using chlorophyll in their leaves.",
        "bad": "Plants eat sunlight.",
    },
]


async def measure_position_bias() -> dict:
    print("\n  --- Position Bias Test ---")
    consistent = 0
    judged = 0
    failures = 0
    for i, p in enumerate(JUDGE_PAIRS, 1):
        try:
            # TODO: Judge the pair twice — good as A, then good as B.
            ab = ____
            ba = ____
        except JudgeParseError as exc:
            failures += 1
            print(f"    Pair {i}: JUDGE FAILURE — {exc}")
            continue
        judged += 1
        # TODO: consistent ONLY if the good answer wins in BOTH orders.
        consistent_this = ____
        consistent += int(consistent_this)
        tag = "consistent" if consistent_this else "POSITION BIAS / WRONG PICK"
        print(f"    Pair {i}: AB={ab['winner']}, BA={ba['winner']}  [{tag}]")
    if judged == 0:
        raise RuntimeError(
            f"The judge produced no usable verdicts ({failures} failures). "
            "Check the model in OLLAMA_CHAT_MODEL follows JSON instructions."
        )
    rate = consistent / judged
    print(f"\n  Position consistency: {consistent}/{judged} judged pairs ({rate:.0%})")
    print(f"  Judge failures:       {failures}/{len(JUDGE_PAIRS)}")
    print(f"  Bias: {'LOW' if rate > 0.7 else 'HIGH'}")
    return {"rate": rate, "judged": judged, "failures": failures}


async def measure_swap_averaged_accuracy() -> dict:
    print("\n  --- Swap-averaged judge (mitigation) ---")
    correct = 0
    judged = 0
    failures = 0
    for i, p in enumerate(JUDGE_PAIRS, 1):
        try:
            v = await swap_averaged_judge(p["prompt"], p["good"], p["bad"])
        except JudgeParseError as exc:
            failures += 1
            print(f"    Pair {i}: JUDGE FAILURE — {exc}")
            continue
        judged += 1
        correct += int(v["winner"] == "x")
        print(
            f"    Pair {i}: good={v['score_x']:.1f}  bad={v['score_y']:.1f}  "
            f"-> {'good' if v['winner'] == 'x' else v['winner']}"
        )
    if judged == 0:
        raise RuntimeError(f"Swap-averaged judge produced no usable verdicts ({failures} failures).")
    accuracy = correct / judged
    print(f"  Swap-averaged accuracy: {correct}/{judged} ({accuracy:.0%}), failures: {failures}")
    return {"accuracy": accuracy, "judged": judged, "failures": failures}



## TASK 5 — Verbosity bias test


Uses the swap-averaged judge so position bias cannot masquerade as
    verbosity bias.


In [ ]:
async def measure_verbosity_bias() -> dict:
    print("\n  --- Verbosity Bias Test ---")
    test_prompt = "What is machine learning?"
    concise = (
        "Machine learning is a subset of AI where algorithms learn patterns from "
        "data to make predictions without explicit programming."
    )
    verbose = (
        "Machine learning is a very interesting and important field of study that "
        "has been gaining a lot of attention in recent years. It is essentially a "
        "subset of artificial intelligence. The basic idea is that instead of "
        "explicitly programming every rule, we let the computer learn from data. "
    ) * 3

    # TODO: Judge concise vs verbose with the swap-averaged judge.
    verdict = ____
    print(f"    Concise ({len(concise)} chars) avg score: {verdict['score_x']:.1f}")
    print(f"    Verbose ({len(verbose)} chars) avg score: {verdict['score_y']:.1f}")
    print(f"    Winner: {'concise' if verdict['winner'] == 'x' else 'verbose' if verdict['winner'] == 'y' else 'tie'}")
    print(f"    Bias: {'VERBOSITY BIAS' if verdict['winner'] == 'y' else 'OK'}")
    return verdict


position_result  = await measure_position_bias()
position_consistency = position_result["rate"]
swap_result  = await measure_swap_averaged_accuracy()
verbosity_verdict  = await measure_verbosity_bias()



### Checkpoint 5


In [ ]:
assert 0.0 <= position_consistency <= 1.0
assert position_result["judged"] + position_result["failures"] == len(JUDGE_PAIRS)
assert 0.0 <= swap_result["accuracy"] <= 1.0
assert verbosity_verdict["winner"] in ("x", "y", "tie")
print("\n✓ Checkpoint 5 passed — LLM-as-judge bias measurements complete\n")



## TASK 6 — Benchmarks survey


lm-eval-harness (EleutherAI):
    Unified evaluation framework for the static benchmarks above
    (MMLU, HellaSwag, HumanEval, TruthfulQA, GSM8K, MBPP, ARC).
    MT-Bench is NOT an lm-eval task: it is a multi-turn set scored by an
    LLM judge — the same technique (and the same biases) as Task 3.
    Install: pip install lm-eval
    Usage:   lm_eval --model hf --model_args pretrained=MODEL \\
                     --tasks mmlu,hellaswag,gsm8k
    Reports: each task's own metric (acc, acc_norm, exact_match, pass@1).

  Pre- vs post-alignment expectation:
    helpfulness & safety should improve after DPO.
    raw knowledge (MMLU) should NOT drop significantly.
    if MMLU drops > 3pp, the policy drifted too far from the reference:
    RAISE beta (stronger KL anchor) or train for fewer steps.


In [ ]:
print("=" * 70)
print("TASK 6: Evaluation Benchmarks Survey")
print("=" * 70)

# TODO: Build a polars DataFrame with 8 rows covering:
#   MMLU, HellaSwag, HumanEval, MT-Bench, TruthfulQA, GSM8K, MBPP, ARC-Challenge.
#   Columns: Benchmark, Domain, Format, Measures.
benchmarks = ____
print(benchmarks)

print(
)



### Checkpoint 6


In [ ]:
assert benchmarks.height >= 8
print("✓ Checkpoint 6 passed — benchmarks survey loaded\n")



## APPLY — Hospital clinical RAG assistant evaluation plan


In [ ]:
# BUSINESS SCENARIO (illustrative): a Singapore public hospital is
# evaluating a DPO-aligned LLM for a clinical RAG assistant used by
# junior doctors. You must define the evaluation plan.
#
# EVALUATION MATRIX:
#   1. Safety    -> swap-averaged LLM-as-judge + verbosity check
#                   (this exercise) on a held-out clinical adversarial set
#   2. Knowledge -> MMLU (medicine subset), a clinical QA benchmark
#   3. Reasoning -> GSM8K for diagnostic math, custom clinical CoT eval
#   4. Honesty   -> TruthfulQA + "I don't know" rate on unanswerable queries
#
# APPROVAL GATE (the hospital's clinical AI governance board, informed
# by IMDA's AI Verify testing framework and HSA guidance on software
# medical devices) — thresholds below are illustrative:
#   - Safety refusal >= 85% on clinical adversarial set
#   - MMLU medicine >= baseline - 2pp (no knowledge degradation)
#   - TruthfulQA >= 80% (no hallucination on unanswerable)
#   - Position-swap consistency >= 75% (judge itself is reliable)

print("=" * 70)
print("APPLICATION — Hospital clinical RAG assistant evaluation plan")
print("=" * 70)

# TODO: Build a 5-row polars DataFrame with columns:
#   Dimension, Method, "Approval Gate", Current
# Cover Safety, Knowledge, Reasoning, Honesty, Judge Quality. Use the
# position_consistency and position_result["failures"] measurements from
# above in the "Current" column of the Judge Quality row.
eval_plan = ____
print(eval_plan)

# Illustrative planning figures (not measurements)
CLINICAL_RAG_USERS = 420  # junior doctors at the hospital
HOURS_SAVED_PER_USER_PER_WEEK = 3.5
DOCTOR_HOURLY_COST_SGD = 90
annual_hours_saved = CLINICAL_RAG_USERS * HOURS_SAVED_PER_USER_PER_WEEK * 52
annual_value_sgd = annual_hours_saved * DOCTOR_HOURLY_COST_SGD

print(f"\n  Target users (junior doctors):  {CLINICAL_RAG_USERS:,}")
print(f"  Hours saved/user/week:          {HOURS_SAVED_PER_USER_PER_WEEK}")
print(f"  Doctor hourly cost:             S${DOCTOR_HOURLY_COST_SGD}")
print(f"  Annual hours saved:             {annual_hours_saved:,.0f}")
print(f"  Annual value of time saved:     S${annual_value_sgd:,.0f}")
print(
    "\n  Value only materialises if ALL approval gates above are met —\n"
    "  otherwise the hospital deploys rule-based triage and the investment is lost."
)



### Checkpoint Application


In [ ]:
assert eval_plan.height == 5
print("\n✓ Application checkpoint passed — hospital evaluation plan ready\n")



## REFLECTION


[x] GRPO intuition: std-normalised group-relative advantages (scale-free)
  [x] Computed the clipped GRPO objective with its KL penalty to pi_ref
  [x] Visualised reward and advantage heatmaps
  [x] LLM-as-judge with Kaizen Delegate — structured JSON verdict, with
      parse failures counted, never turned into fake ties
  [x] Measured position bias via A/B swap and mitigated it with a
      swap-averaged judge
  [x] Measured verbosity bias with concise-vs-padded test
  [x] Surveyed the standard LLM benchmarks (MMLU, HellaSwag, HumanEval,
      MT-Bench, TruthfulQA, GSM8K, MBPP, ARC)
  [x] Drafted a hospital clinical RAG evaluation plan with approval gates
      and quantified business value

  KEY INSIGHT: Evaluation IS the product. An LLM without an evaluation
  plan is a prototype, not a system. DPO trains the behaviour; evaluation
  proves the behaviour. For regulated deployments (healthcare, finance),
  the evaluation plan is what gets signed off, not the model weights.

  NEXT: Exercise 4 (RAG) grounds LLM responses in retrieved documents.
  Instead of relying on training data alone, RAG retrieves relevant
  text at inference time — enabling up-to-date, verifiable answers.


In [ ]:
print("=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)

